# 11 — Sub-Agents：子代理模式

主代理遇到复杂的跨文件任务时，可以派出专注的子代理处理。子代理有独立的工具白名单和最大回合数，不会和主代理互相干扰。

| 问题 | 子代理怎么解决 |
|------|----------------|
| 主代理 context 被大量文件读取占满 | 子代理有独立 context，调查完只返回摘要 |
| 读写工具混在一起，reviewer 误调用写操作 | 子代理只注册 `allowed_tools` 白名单里的工具 |
| 主任务被子任务的细节淹没 | 子代理结束后只返回一个 `ToolResult`，主代理继续 |

## 什么时候用子代理

- 任务需要大量读文件（会占满主代理的 context）
- 需要专注分析（reviewer 不需要写文件工具）
- 主代理 context 快满了，派子代理去做一个子任务

## ASCII 图解

```
主代理 context (20k tokens)
    │
    ├─ 派出 CodebaseInvestigator (独立 context，只有读工具)
    │      └─ 调查了 50 个文件，消耗 80k tokens
    │      └─ 返回摘要 (2k tokens)
    │
    ├─ 收到摘要，继续主任务
    └─ context 依然干净
```

子代理本身就是一个 `Tool`——主代理把它注册到 `ToolRegistry`，LLM 就能自主决定什么时候调用它。

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from dataclasses import dataclass, field
from typing import AsyncGenerator
import asyncio
import json

from src.llm_client import LLMClient
from src.context_manager import ContextManager, build_system_prompt
from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry

print("导入成功")

## 1. SubAgentConfig 数据类

所有子代理的配置集中在一个 dataclass 里。设计原则：

- `allowed_tools`：工具名称白名单，子代理只能用这些工具，多一个都不行
- `max_turns`：防止子代理陷入死循环，每个子代理有独立的上限
- `timeout`：整个子任务的时间上限，防止一个慢子代理卡死主代理

In [ ]:
@dataclass
class SubAgentConfig:
    name: str                          # 子代理名称（同时也是 Tool.name）
    system_prompt: str                 # 专用系统 prompt，定义子代理的角色和约束
    allowed_tools: list[str]           # 工具白名单（tool.name 列表）
    description: str = ""              # 工具描述，LLM 据此决定何时调用
    max_turns: int = 5                 # 最大回合数
    timeout: float = 120.0             # 整个子任务的超时秒数


# 验证数据类
config = SubAgentConfig(
    name="investigate_codebase",
    system_prompt="你是一个代码库分析师。",
    allowed_tools=["read_file", "list_directory", "glob"],
    description="调查代码库的结构和内容。",
    max_turns=8,
    timeout=120.0,
)
print(config)

## 2. Agentic Loop

子代理内部需要运行一个工具调用循环（和主代理一样）。先实现这个通用的 `agentic_loop` 函数，主代理和子代理都复用它。

循环逻辑：

```
用户消息 → LLM → tool_calls?
                    ├─ 是 → 执行工具 → 把结果加入 context → 回到 LLM
                    └─ 否 → 返回最终回答
```

`LoopEvent` 用于在循环里传递事件，调用方可以选择监听哪些事件（流式输出、工具调用记录、最终完成）。

In [ ]:
from dataclasses import dataclass as dc
from enum import Enum


class EventKind(Enum):
    TOOL_CALL = "tool_call"        # LLM 决定调用工具
    TOOL_RESULT = "tool_result"    # 工具执行完毕
    TEXT_DELTA = "text_delta"      # 流式文本片段（如果启用流式）
    TEXT_COMPLETE = "text_complete" # 本轮 LLM 回复完整文本
    LOOP_END = "loop_end"          # 整个循环结束
    ERROR = "error"                # 出错


@dc
class LoopEvent:
    kind: EventKind
    data: dict  # 事件附带的数据


async def agentic_loop(
    task: str,
    llm: LLMClient,
    ctx: ContextManager,
    registry: ToolRegistry,
    max_turns: int = 10,
) -> AsyncGenerator[LoopEvent, None]:
    """
    通用 Agentic Loop。

    把 task 加入 context，然后循环让 LLM 决定调用工具或结束。
    每次有意义的事件都 yield 出来，调用方可以按需消费。

    参数：
        task      - 用户输入的任务文本
        llm       - LLM 客户端（不新建，复用父代理的连接）
        ctx       - 独立的 ContextManager（子代理有自己的）
        registry  - 独立的 ToolRegistry（子代理有自己的工具集）
        max_turns - 最大回合数
    """
    # 把任务加入 context
    ctx.add_user_message(task)

    for turn in range(max_turns):
        # 调用 LLM，带上当前 context 和可用工具
        try:
            response_text, usage = await llm.chat_completion(
                messages=ctx.get_messages(),
                tools=registry.get_schemas() if registry.list_tools() else None,
            )
        except Exception as e:
            yield LoopEvent(kind=EventKind.ERROR, data={"error": str(e), "turn": turn})
            return

        ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)

        # response_text 可能是纯文本，也可能是 JSON 格式的 tool_calls
        # LLMClient.chat_completion 返回 (str, TokenUsage)
        # 这里约定：如果 LLM 返回的是工具调用，response_text 是 JSON
        # 实际上 LLMClient 内部解析了 tool_calls，需要拿原始响应
        # 为了演示清晰，这里用简化版：直接解析 response_text

        # 尝试解析 tool_calls
        tool_calls = None
        content_text = response_text

        # 检查是否有工具调用（LLM 返回结构化数据时）
        if isinstance(response_text, str) and response_text.strip().startswith("["):
            try:
                possible_calls = json.loads(response_text)
                if isinstance(possible_calls, list) and all(
                    "function" in c or "name" in c for c in possible_calls
                ):
                    tool_calls = possible_calls
                    content_text = ""
            except json.JSONDecodeError:
                pass

        if tool_calls:
            # LLM 决定调用工具
            ctx.add_assistant_message(content=content_text, tool_calls=tool_calls)

            for tc in tool_calls:
                # 兼容两种 tool_call 格式
                if "function" in tc:
                    tool_name = tc["function"]["name"]
                    params_raw = tc["function"].get("arguments", "{}")
                else:
                    tool_name = tc.get("name", "")
                    params_raw = tc.get("arguments", "{}")

                tc_id = tc.get("id", f"call_{turn}")

                if isinstance(params_raw, str):
                    try:
                        params = json.loads(params_raw)
                    except json.JSONDecodeError:
                        params = {}
                else:
                    params = params_raw

                yield LoopEvent(kind=EventKind.TOOL_CALL, data={
                    "tool": tool_name, "params": params, "id": tc_id
                })

                # 执行工具
                result = await registry.invoke(tool_name, params)
                ctx.add_tool_result(tc_id, result.content)

                yield LoopEvent(kind=EventKind.TOOL_RESULT, data={
                    "tool": tool_name,
                    "success": result.success,
                    "content": result.content[:200],  # 摘要，避免事件数据过大
                    "id": tc_id,
                })
        else:
            # LLM 返回了最终文本回答
            ctx.add_assistant_message(content=content_text)
            yield LoopEvent(kind=EventKind.TEXT_COMPLETE, data={"text": content_text})
            break
    else:
        # 达到最大回合数
        yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "max_turns_reached"})
        return

    yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "completed"})


print("agentic_loop 定义完成")

## 3. 适配真实的 LLMClient

上面的 `agentic_loop` 是简化版。实际上 `LLMClient.chat_completion` 返回 `(str, TokenUsage)`，工具调用的解析需要访问原始的 OpenAI response。

下面实现一个基于真实 OpenAI 协议的版本，直接用 `openai.AsyncOpenAI` 客户端，并处理 `tool_calls` 字段。

In [ ]:
from openai import AsyncOpenAI


async def agentic_loop_v2(
    task: str,
    raw_client: AsyncOpenAI,
    model: str,
    ctx: ContextManager,
    registry: ToolRegistry,
    max_turns: int = 10,
) -> AsyncGenerator[LoopEvent, None]:
    """
    基于真实 OpenAI 协议的 Agentic Loop。

    直接使用 openai.AsyncOpenAI，正确处理 tool_calls 字段。
    这是子代理实际使用的版本。
    """
    ctx.add_user_message(task)
    schemas = registry.get_schemas() if registry.list_tools() else None

    for turn in range(max_turns):
        try:
            response = await raw_client.chat.completions.create(
                model=model,
                messages=ctx.get_messages(),
                tools=schemas,
            )
        except Exception as e:
            yield LoopEvent(kind=EventKind.ERROR, data={"error": str(e), "turn": turn})
            return

        # 更新 token 用量
        if response.usage:
            ctx.update_usage(
                response.usage.prompt_tokens,
                response.usage.completion_tokens,
            )

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls" and msg.tool_calls:
            # LLM 决定调用工具
            tool_calls_dicts = [tc.model_dump() for tc in msg.tool_calls]
            ctx.add_assistant_message(
                content=msg.content or "",
                tool_calls=tool_calls_dicts,
            )

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                try:
                    params = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    params = {}

                yield LoopEvent(kind=EventKind.TOOL_CALL, data={
                    "tool": tool_name, "params": params, "id": tc.id
                })

                result = await registry.invoke(tool_name, params)
                ctx.add_tool_result(tc.id, result.content)

                yield LoopEvent(kind=EventKind.TOOL_RESULT, data={
                    "tool": tool_name,
                    "success": result.success,
                    "content_preview": result.content[:300],
                    "id": tc.id,
                })
        else:
            # 返回最终文本
            final_text = msg.content or ""
            ctx.add_assistant_message(content=final_text)
            yield LoopEvent(kind=EventKind.TEXT_COMPLETE, data={"text": final_text})
            break
    else:
        yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "max_turns_reached"})
        return

    yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "completed"})


print("agentic_loop_v2 定义完成")

## 4. SubAgent 类

子代理是一个 `Tool`，主代理把它注册到 `ToolRegistry` 后，LLM 就能像调用普通工具一样调用它。

`execute` 方法内部：
1. 创建独立的 `ContextManager`（子代理的 system_prompt）
2. 创建独立的 `ToolRegistry`，只注册 `allowed_tools` 中的工具
3. 从父代理的 registry 中筛选出允许的工具
4. 运行 `agentic_loop_v2`，收集 `TEXT_COMPLETE` 事件
5. 返回 `ToolResult.ok(最终回答)`

```
SubAgent.execute(task)
    │
    ├─ 创建 sub_ctx (SubAgentConfig.system_prompt)
    ├─ 创建 sub_registry (只含 allowed_tools)
    │
    └─ agentic_loop_v2(task, raw_client, sub_ctx, sub_registry)
           ├─ turn 1: read_file → 结果加入 sub_ctx
           ├─ turn 2: glob → 结果加入 sub_ctx  
           └─ turn N: 最终文本 → TEXT_COMPLETE
    │
    └─ 返回 ToolResult.ok(最终文本)
```

In [ ]:
class SubAgent(Tool):
    """
    子代理：封装成 Tool，可以注册到主代理的 ToolRegistry。

    子代理有独立的 context 和工具集，不污染主代理的状态。
    主代理从 LLM 的角度来看，调用子代理和调用 read_file 没有区别——
    都是传一个参数，拿回一个结果。
    """

    def __init__(
        self,
        config: SubAgentConfig,
        raw_client: AsyncOpenAI,
        model: str,
        parent_registry: ToolRegistry,
        parent_cwd: Path = Path("."),
    ):
        self._config = config
        self._raw_client = raw_client
        self._model = model
        self._parent_registry = parent_registry  # 从这里筛选 allowed_tools
        self._cwd = parent_cwd.resolve()

    # ── Tool 接口 ────────────────────────────────────────────────────────────

    @property
    def name(self) -> str:
        return self._config.name

    @property
    def description(self) -> str:
        return self._config.description or self._config.system_prompt[:80]

    @property
    def kind(self) -> ToolKind:
        # 子代理可能读也可能写，统一用 READ 表示它是「信息收集」操作
        # 如果子代理有写工具，可以改成 WRITE
        return ToolKind.READ

    def schema(self) -> dict:
        """子代理只暴露一个参数 task，主代理告诉它做什么就行。"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "task": {
                            "type": "string",
                            "description": "交给子代理的具体任务描述。越清晰越好。"
                        }
                    },
                    "required": ["task"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        if "task" not in params or not params["task"].strip():
            return ["缺少必填参数: task"]
        return []

    async def execute(self, params: dict) -> ToolResult:
        task = params["task"]

        # 1. 创建子代理独立的 context
        sub_ctx = ContextManager(system_prompt=self._config.system_prompt)

        # 2. 创建子代理独立的工具集（只含 allowed_tools）
        sub_registry = ToolRegistry()
        missing_tools = []
        for tool_name in self._config.allowed_tools:
            tool = self._parent_registry.get(tool_name)
            if tool:
                sub_registry.register(tool)
            else:
                missing_tools.append(tool_name)

        if missing_tools:
            # 父代理没有注册这些工具，记录警告但不中断
            # 子代理会在调用时才发现工具不存在
            print(f"[SubAgent:{self.name}] 警告：父注册表中找不到工具: {missing_tools}")

        # 3. 运行 agentic_loop，收集结果
        final_texts = []
        tool_call_count = 0

        try:
            loop_gen = agentic_loop_v2(
                task=task,
                raw_client=self._raw_client,
                model=self._model,
                ctx=sub_ctx,
                registry=sub_registry,
                max_turns=self._config.max_turns,
            )

            async with asyncio.timeout(self._config.timeout):
                async for event in loop_gen:
                    if event.kind == EventKind.TOOL_CALL:
                        tool_call_count += 1
                        print(f"  [子代理:{self.name}] 调用工具: {event.data['tool']}")
                    elif event.kind == EventKind.TEXT_COMPLETE:
                        final_texts.append(event.data["text"])
                    elif event.kind == EventKind.ERROR:
                        return ToolResult.error(
                            f"子代理出错: {event.data['error']}",
                            agent=self.name,
                        )

        except asyncio.TimeoutError:
            return ToolResult.error(
                f"子代理 '{self.name}' 超时（>{self._config.timeout}s）。\n"
                f"已完成 {tool_call_count} 次工具调用。",
                agent=self.name,
                tool_calls=tool_call_count,
            )
        except Exception as e:
            return ToolResult.error(
                f"子代理 '{self.name}' 异常: {type(e).__name__}: {e}",
                agent=self.name,
            )

        # 4. 返回最终回答
        final_answer = "\n\n".join(final_texts) if final_texts else "（子代理未返回任何内容）"

        return ToolResult.ok(
            final_answer,
            agent=self.name,
            tool_calls=tool_call_count,
            context_tokens=sub_ctx.estimated_tokens,
        )


print("SubAgent 类定义完成")

## 5. 内置子代理：CodebaseInvestigator 和 CodeReviewer

两个开箱即用的子代理，覆盖最常见的使用场景。

**CodebaseInvestigator**：只读工具（`read_file`、`list_directory`、`glob`），专门用于调查代码库结构。主代理 context 快满时派它出去读文件，只返回摘要。

**CodeReviewer**：只读工具（`read_file`、`glob`），不能写文件，确保 review 过程只读不改。

In [ ]:
def create_codebase_investigator(
    raw_client: AsyncOpenAI,
    model: str,
    parent_registry: ToolRegistry,
    cwd: Path = Path("."),
) -> SubAgent:
    """
    代码库调查员。
    负责调查代码库结构，找到特定功能的实现位置。
    只能读文件，不能修改任何内容。
    """
    config = SubAgentConfig(
        name="investigate_codebase",
        description=(
            "调查代码库的结构和内容。"
            "用于了解项目架构、找到特定功能的实现位置、统计文件数量等。"
            "返回简洁的调查报告。"
        ),
        system_prompt=(
            "你是一个代码库分析师。你的任务是调查代码库，回答关于代码的问题。\n"
            "你只能读文件，不能修改任何东西。\n"
            "调查完成后，用简洁的报告格式返回结果：\n"
            "- 先列出目录结构（一两层即可）\n"
            "- 再描述每个主要模块的职责\n"
            "- 最后回答用户的具体问题\n"
            "保持简洁，不要复制粘贴大段代码。"
        ),
        allowed_tools=["read_file", "list_directory", "glob"],
        max_turns=8,
        timeout=120.0,
    )
    return SubAgent(config, raw_client, model, parent_registry, cwd)


def create_code_reviewer(
    raw_client: AsyncOpenAI,
    model: str,
    parent_registry: ToolRegistry,
    cwd: Path = Path("."),
) -> SubAgent:
    """
    代码审查员。
    对指定文件或代码进行 code review，找出潜在问题和改进建议。
    只能读文件，不能修改。
    """
    config = SubAgentConfig(
        name="review_code",
        description=(
            "对指定文件或代码进行 code review，找出潜在问题和改进建议。"
            "传入要审查的文件路径或模块名称。"
            "返回结构化的审查报告。"
        ),
        system_prompt=(
            "你是一个严格的代码审查员。分析代码质量、潜在 bug、安全问题、性能问题。\n"
            "返回结构化的审查报告，格式如下：\n"
            "\n"
            "## [严重问题]\n"
            "（会导致 bug 或安全漏洞的问题）\n"
            "\n"
            "## [改进建议]\n"
            "（可以改进但不影响功能的地方）\n"
            "\n"
            "## [代码亮点]\n"
            "（写得好的地方，值得保留或推广）\n"
            "\n"
            "每个条目都要指出具体的文件和行号。如果代码没有问题，直接说'未发现严重问题'。"
        ),
        allowed_tools=["read_file", "glob"],
        max_turns=5,
        timeout=90.0,
    )
    return SubAgent(config, raw_client, model, parent_registry, cwd)


print("工厂函数定义完成")
print("  create_codebase_investigator -> allowed_tools: ['read_file', 'list_directory', 'glob']")
print("  create_code_reviewer         -> allowed_tools: ['read_file', 'glob']")

## 6. 注册子代理到主代理的 ToolRegistry

子代理本身就是 `Tool`，注册方式和普通工具完全一样。
注册后，LLM 在主代理的 `get_schemas()` 里能看到子代理的 schema，自主决定什么时候调用。

In [ ]:
# 先注册基础工具（文件工具）
# 由于 src/ 目录不存在，这里用内联版本演示注册流程
# 实际项目里从 src.file_tools 导入
from pathlib import Path
import os

cwd = Path(".").resolve()

# 创建一个模拟的文件工具（用于演示注册流程）
class MockReadFileTool(Tool):
    def __init__(self, cwd):
        self.cwd = cwd
    @property
    def name(self): return "read_file"
    @property
    def description(self): return "读取文件内容"
    @property
    def kind(self): return ToolKind.READ
    def schema(self):
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}
        }}
    def validate(self, params): return []
    async def execute(self, params):
        try:
            p = self.cwd / params["path"]
            content = p.read_text()
            return ToolResult.ok(content)
        except Exception as e:
            return ToolResult.error(str(e))

class MockListDirTool(Tool):
    def __init__(self, cwd):
        self.cwd = cwd
    @property
    def name(self): return "list_directory"
    @property
    def description(self): return "列出目录内容"
    @property
    def kind(self): return ToolKind.READ
    def schema(self):
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": {"type": "object", "properties": {"path": {"type": "string"}}}
        }}
    def validate(self, params): return []
    async def execute(self, params):
        path = self.cwd / params.get("path", ".")
        try:
            entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
            lines = [e.name + ("/" if e.is_dir() else "") for e in entries if not e.name.startswith(".")]
            return ToolResult.ok("\n".join(lines))
        except Exception as e:
            return ToolResult.error(str(e))

class MockGlobTool(Tool):
    def __init__(self, cwd):
        self.cwd = cwd
    @property
    def name(self): return "glob"
    @property
    def description(self): return "按 glob 模式搜索文件"
    @property
    def kind(self): return ToolKind.READ
    def schema(self):
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": {"type": "object", "properties": {"pattern": {"type": "string"}}, "required": ["pattern"]}
        }}
    def validate(self, params): return []
    async def execute(self, params):
        matches = [str(p.relative_to(self.cwd)) for p in self.cwd.glob(params["pattern"]) if p.is_file()]
        return ToolResult.ok("\n".join(sorted(matches)) or "无匹配文件")


# 主代理的 ToolRegistry
main_registry = ToolRegistry()
main_registry.register(MockReadFileTool(cwd))
main_registry.register(MockListDirTool(cwd))
main_registry.register(MockGlobTool(cwd))

# 初始化 OpenAI 客户端（复用，不新建连接）
raw_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "gpt-oss:120b"

# 注册子代理到主 registry
main_registry.register(create_codebase_investigator(raw_client, MODEL, main_registry, cwd))
main_registry.register(create_code_reviewer(raw_client, MODEL, main_registry, cwd))

print("主代理工具列表:")
for t in main_registry.list_tools():
    icon = "[子代理]" if isinstance(t, SubAgent) else "[工具  ]"
    print(f"  {icon} {t.name:30s} | kind={t.kind.value}")

## 7. 单元测试：子代理独立执行

先不涉及主代理，直接测试子代理能否独立完成一个调查任务。

In [ ]:
# 测试 CodebaseInvestigator：调查当前 course/ 目录
investigator = create_codebase_investigator(raw_client, MODEL, main_registry, cwd)

print(f"子代理名称: {investigator.name}")
print(f"schema: {json.dumps(investigator.schema()['function']['parameters'], ensure_ascii=False, indent=2)}")
print()
print(f"allowed_tools: {investigator._config.allowed_tools}")
print(f"max_turns: {investigator._config.max_turns}")
print(f"timeout: {investigator._config.timeout}s")

In [ ]:
# 直接调用子代理（不通过主代理）
print("直接调用 investigate_codebase 子代理...")
print("="*60)

result = await investigator.execute({
    "task": "列出当前目录的结构，告诉我有哪些文件，这些文件大概是做什么用的。"
})

print(f"\n子代理执行结果:")
print(f"  success: {result.success}")
print(f"  调用了 {result.metadata.get('tool_calls', 0)} 次工具")
print(f"  子代理 context 消耗: {result.metadata.get('context_tokens', 0)} tokens")
print()
print("子代理返回的报告:")
print("-"*60)
print(result.content)

In [ ]:
# 测试错误处理：传空 task
print("测试空 task:")
errors = investigator.validate({"task": ""})
print(f"  validate 错误: {errors}")
print()

# 测试 allowed_tools 白名单：子代理不应该能调用 write_file
print("测试工具白名单:")
# 在子代理的 sub_registry 里查找 write_file（不应该存在）
sub_registry_test = ToolRegistry()
for tool_name in investigator._config.allowed_tools:
    tool = main_registry.get(tool_name)
    if tool:
        sub_registry_test.register(tool)

available = [t.name for t in sub_registry_test.list_tools()]
print(f"  子代理可用工具: {available}")
print(f"  write_file 在白名单中: {'write_file' in available}")
print(f"  read_file 在白名单中: {'read_file' in available}")

## 8. 完整流程：主代理调用子代理

演示完整流程：
1. 给主代理一个任务：「调查一下当前目录里有哪些 Notebook，每个的主要内容是什么？」
2. 主代理决定调用 `investigate_codebase` 子代理
3. 子代理读文件，返回分析报告
4. 主代理根据报告回答用户

```
用户 → 主代理 → [调用 investigate_codebase(task)] → 子代理
                                                        ├─ list_directory
                                                        ├─ read_file(01_...)  
                                                        └─ 返回报告
         ← 收到报告 ← ──────────────────────────────────
         → 给用户最终回答
```

In [ ]:
# 主代理的 context（含子代理工具）
main_ctx = ContextManager(
    system_prompt=(
        "你是一个智能助手。你有以下工具可用：\n"
        "- read_file, list_directory, glob：文件系统工具\n"
        "- investigate_codebase：代码库调查子代理，调查整个代码库时优先使用它\n"
        "- review_code：代码审查子代理，需要做 code review 时使用它\n"
        "\n"
        "当任务需要调查大量文件时，主动使用 investigate_codebase 子代理，\n"
        "这样可以避免消耗主对话的 context。"
    )
)

user_task = "调查一下当前目录里有哪些 Notebook，每个的主要内容是什么？用一句话概括每个文件。"
print(f"用户任务: {user_task}")
print("="*60)

# 运行主代理
turn_count = 0
async for event in agentic_loop_v2(
    task=user_task,
    raw_client=raw_client,
    model=MODEL,
    ctx=main_ctx,
    registry=main_registry,
    max_turns=5,
):
    if event.kind == EventKind.TOOL_CALL:
        turn_count += 1
        print(f"\n[主代理 Turn {turn_count}] 调用工具: {event.data['tool']}")
        task_param = event.data['params'].get('task', '')
        if task_param:
            print(f"  子代理任务: {task_param[:100]}..." if len(task_param) > 100 else f"  子代理任务: {task_param}")
    elif event.kind == EventKind.TOOL_RESULT:
        preview = event.data.get('content_preview', '')[:150]
        print(f"  工具结果 (前150字符): {preview}")
    elif event.kind == EventKind.TEXT_COMPLETE:
        print(f"\n主代理最终回答:")
        print("-"*60)
        print(event.data['text'])
    elif event.kind == EventKind.LOOP_END:
        print(f"\n循环结束: {event.data['reason']}")

print(f"\n主代理 context 消耗: {main_ctx.estimated_tokens} tokens")

## 9. 错误场景测试

子代理要处理的几种失败情况：

- 父 registry 中找不到 `allowed_tools` 里的某个工具
- 子代理内部 LLM 报错
- 超时

In [ ]:
# 场景 1：allowed_tools 中有父 registry 里不存在的工具
print("场景 1：allowed_tools 包含不存在的工具")
print("-"*50)

config_with_missing = SubAgentConfig(
    name="test_missing_tools",
    system_prompt="测试用",
    description="测试工具缺失",
    allowed_tools=["read_file", "nonexistent_tool", "another_missing"],
    max_turns=2,
)

# 创建时不报错，但执行时会打印警告并跳过缺失工具
agent_missing = SubAgent(config_with_missing, raw_client, MODEL, main_registry, cwd)
result = await agent_missing.execute({"task": "列出当前目录"})

print(f"执行结果: success={result.success}")
print(f"（警告信息应该在上方打印）")
print()

In [ ]:
# 场景 2：超时测试（设一个极短的 timeout）
print("场景 2：子代理超时")
print("-"*50)

config_timeout = SubAgentConfig(
    name="test_timeout",
    system_prompt="你是一个故意很慢的代理。每次都调用很多工具。",
    description="超时测试",
    allowed_tools=["list_directory", "read_file", "glob"],
    max_turns=10,
    timeout=0.001,  # 1 毫秒，必然超时
)

agent_timeout = SubAgent(config_timeout, raw_client, MODEL, main_registry, cwd)
result = await agent_timeout.execute({"task": "列出所有文件并读取每一个"})

print(f"执行结果: success={result.success}")
print(f"错误信息: {result.content[:200]}")

In [ ]:
# 场景 3：子代理在空目录中工作（找不到任何文件）
import tempfile

print("场景 3：空目录中的调查")
print("-"*50)

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    
    # 创建一个只有空目录的 registry
    empty_registry = ToolRegistry()
    empty_registry.register(MockReadFileTool(tmp))
    empty_registry.register(MockListDirTool(tmp))
    empty_registry.register(MockGlobTool(tmp))
    
    agent_empty = create_codebase_investigator(raw_client, MODEL, empty_registry, tmp)
    result = await agent_empty.execute({"task": "这个目录里有什么文件？"})
    
    print(f"执行结果: success={result.success}")
    print(f"调用了 {result.metadata.get('tool_calls', 0)} 次工具")
    print(f"回答: {result.content[:300]}")

## 10. 子代理的 context 隔离验证

子代理最重要的特性是 context 隔离：子代理读了大量文件，主代理的 context 不受影响。

In [ ]:
# 验证 context 隔离
print("context 隔离验证")
print("="*60)

# 主代理 context（只有初始 system prompt + 一条用户消息）
isolation_ctx = ContextManager(system_prompt="你是助手。")
isolation_ctx.add_user_message("调查代码库")

tokens_before = isolation_ctx.estimated_tokens
print(f"调用子代理前，主代理 context tokens: {tokens_before}")

# 模拟子代理执行（创建独立 context）
sub_ctx_test = ContextManager(system_prompt="你是代码分析师。")
sub_ctx_test.add_user_message("调查代码库")
# 模拟子代理读了很多文件
for i in range(5):
    sub_ctx_test.add_assistant_message("", tool_calls=[{"id": f"c{i}", "type": "function", "function": {"name": "read_file", "arguments": '{"path": "test.py"}'}}])
    sub_ctx_test.add_tool_result(f"c{i}", "x" * 500)  # 模拟读了 500 字符的文件

print(f"子代理 context tokens: {sub_ctx_test.estimated_tokens}")

# 子代理执行完，主代理只收到一个 ToolResult
# 假设子代理返回了 200 字符的摘要
isolation_ctx.add_assistant_message("", tool_calls=[{"id": "sa1", "type": "function", "function": {"name": "investigate_codebase", "arguments": '{"task": "调查"}'}}])
isolation_ctx.add_tool_result("sa1", "摘要: 共 5 个文件，主要是工具和测试代码。" * 5)  # ~200 字符

tokens_after = isolation_ctx.estimated_tokens
print(f"子代理返回后，主代理 context tokens: {tokens_after}")
print()
print(f"主代理增加了 {tokens_after - tokens_before} tokens（只有摘要）")
print(f"子代理消耗了 {sub_ctx_test.estimated_tokens} tokens（完全隔离）")
print(f"节省比例: {1 - (tokens_after - tokens_before) / sub_ctx_test.estimated_tokens:.0%}")

## 11. 保存到 src/

In [ ]:
import os

# 确保 src/ 目录存在
os.makedirs("../src", exist_ok=True)

sub_agents_code = '''
# sub_agents.py — 子代理模式
# 主代理可以派出专注的子代理处理复杂的子任务，子代理有独立的 context 和工具集。

from __future__ import annotations

import asyncio
import json
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import AsyncGenerator

from openai import AsyncOpenAI


# ── 事件系统 ─────────────────────────────────────────────────────────────────

class EventKind(Enum):
    TOOL_CALL = "tool_call"
    TOOL_RESULT = "tool_result"
    TEXT_DELTA = "text_delta"
    TEXT_COMPLETE = "text_complete"
    LOOP_END = "loop_end"
    ERROR = "error"


@dataclass
class LoopEvent:
    kind: EventKind
    data: dict


# ── SubAgentConfig ────────────────────────────────────────────────────────────

@dataclass
class SubAgentConfig:
    name: str                          # 子代理名称（同时也是 Tool.name）
    system_prompt: str                 # 专用系统 prompt
    allowed_tools: list[str]           # 工具白名单（tool.name 列表）
    description: str = ""              # 工具描述
    max_turns: int = 5                 # 最大回合数
    timeout: float = 120.0             # 整个子任务的超时秒数


# ── Agentic Loop ──────────────────────────────────────────────────────────────

async def agentic_loop_v2(
    task: str,
    raw_client: AsyncOpenAI,
    model: str,
    ctx,  # ContextManager
    registry,  # ToolRegistry
    max_turns: int = 10,
) -> AsyncGenerator[LoopEvent, None]:
    """通用 Agentic Loop（基于 OpenAI 协议）。"""
    ctx.add_user_message(task)
    schemas = registry.get_schemas() if registry.list_tools() else None

    for turn in range(max_turns):
        try:
            response = await raw_client.chat.completions.create(
                model=model,
                messages=ctx.get_messages(),
                tools=schemas,
            )
        except Exception as e:
            yield LoopEvent(kind=EventKind.ERROR, data={"error": str(e), "turn": turn})
            return

        if response.usage:
            ctx.update_usage(
                response.usage.prompt_tokens,
                response.usage.completion_tokens,
            )

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls" and msg.tool_calls:
            tool_calls_dicts = [tc.model_dump() for tc in msg.tool_calls]
            ctx.add_assistant_message(
                content=msg.content or "",
                tool_calls=tool_calls_dicts,
            )

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                try:
                    params = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    params = {}

                yield LoopEvent(kind=EventKind.TOOL_CALL, data={
                    "tool": tool_name, "params": params, "id": tc.id
                })

                result = await registry.invoke(tool_name, params)
                ctx.add_tool_result(tc.id, result.content)

                yield LoopEvent(kind=EventKind.TOOL_RESULT, data={
                    "tool": tool_name,
                    "success": result.success,
                    "content_preview": result.content[:300],
                    "id": tc.id,
                })
        else:
            final_text = msg.content or ""
            ctx.add_assistant_message(content=final_text)
            yield LoopEvent(kind=EventKind.TEXT_COMPLETE, data={"text": final_text})
            break
    else:
        yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "max_turns_reached"})
        return

    yield LoopEvent(kind=EventKind.LOOP_END, data={"reason": "completed"})


# ── SubAgent ──────────────────────────────────────────────────────────────────

class SubAgent:
    """
    子代理：封装成 Tool 接口，可以注册到主代理的 ToolRegistry。
    子代理有独立的 context 和工具集，不污染主代理的状态。
    """

    def __init__(
        self,
        config: SubAgentConfig,
        raw_client: AsyncOpenAI,
        model: str,
        parent_registry,  # ToolRegistry
        parent_cwd: Path = Path("."),
    ):
        self._config = config
        self._raw_client = raw_client
        self._model = model
        self._parent_registry = parent_registry
        self._cwd = parent_cwd.resolve()

    @property
    def name(self) -> str:
        return self._config.name

    @property
    def description(self) -> str:
        return self._config.description or self._config.system_prompt[:80]

    @property
    def kind(self):
        from src.tool_framework import ToolKind
        return ToolKind.READ

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "task": {
                            "type": "string",
                            "description": "交给子代理的具体任务描述。越清晰越好。"
                        }
                    },
                    "required": ["task"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        if "task" not in params or not params["task"].strip():
            return ["缺少必填参数: task"]
        return []

    def is_mutating(self) -> bool:
        return False

    async def execute(self, params: dict):
        from src.context_manager import ContextManager
        from src.tool_framework import ToolRegistry, ToolResult

        task = params["task"]
        sub_ctx = ContextManager(system_prompt=self._config.system_prompt)

        sub_registry = ToolRegistry()
        missing_tools = []
        for tool_name in self._config.allowed_tools:
            tool = self._parent_registry.get(tool_name)
            if tool:
                sub_registry.register(tool)
            else:
                missing_tools.append(tool_name)

        if missing_tools:
            print(f"[SubAgent:{self.name}] 警告：父注册表中找不到工具: {missing_tools}")

        final_texts = []
        tool_call_count = 0

        try:
            loop_gen = agentic_loop_v2(
                task=task,
                raw_client=self._raw_client,
                model=self._model,
                ctx=sub_ctx,
                registry=sub_registry,
                max_turns=self._config.max_turns,
            )

            async with asyncio.timeout(self._config.timeout):
                async for event in loop_gen:
                    if event.kind == EventKind.TOOL_CALL:
                        tool_call_count += 1
                        print(f"  [子代理:{self.name}] 调用工具: {event.data[\"tool\"]}")
                    elif event.kind == EventKind.TEXT_COMPLETE:
                        final_texts.append(event.data["text"])
                    elif event.kind == EventKind.ERROR:
                        return ToolResult.error(
                            f"子代理出错: {event.data[\"error\"]}",
                            agent=self.name,
                        )

        except asyncio.TimeoutError:
            return ToolResult.error(
                f"子代理 \'{self.name}\' 超时（>{self._config.timeout}s）。\n"
                f"已完成 {tool_call_count} 次工具调用。",
                agent=self.name,
                tool_calls=tool_call_count,
            )
        except Exception as e:
            return ToolResult.error(
                f"子代理 \'{self.name}\' 异常: {type(e).__name__}: {e}",
                agent=self.name,
            )

        final_answer = "\\n\\n".join(final_texts) if final_texts else "（子代理未返回任何内容）"

        return ToolResult.ok(
            final_answer,
            agent=self.name,
            tool_calls=tool_call_count,
            context_tokens=sub_ctx.estimated_tokens,
        )


# ── 内置子代理工厂函数 ─────────────────────────────────────────────────────────

def create_codebase_investigator(
    raw_client: AsyncOpenAI,
    model: str,
    parent_registry,
    cwd: Path = Path("."),
) -> SubAgent:
    config = SubAgentConfig(
        name="investigate_codebase",
        description=(
            "调查代码库的结构和内容。"
            "用于了解项目架构、找到特定功能的实现位置、统计文件数量等。"
            "返回简洁的调查报告。"
        ),
        system_prompt=(
            "你是一个代码库分析师。你的任务是调查代码库，回答关于代码的问题。\n"
            "你只能读文件，不能修改任何东西。\n"
            "调查完成后，用简洁的报告格式返回结果：\n"
            "- 先列出目录结构（一两层即可）\n"
            "- 再描述每个主要模块的职责\n"
            "- 最后回答用户的具体问题\n"
            "保持简洁，不要复制粘贴大段代码。"
        ),
        allowed_tools=["read_file", "list_directory", "glob"],
        max_turns=8,
        timeout=120.0,
    )
    return SubAgent(config, raw_client, model, parent_registry, cwd)


def create_code_reviewer(
    raw_client: AsyncOpenAI,
    model: str,
    parent_registry,
    cwd: Path = Path("."),
) -> SubAgent:
    config = SubAgentConfig(
        name="review_code",
        description=(
            "对指定文件或代码进行 code review，找出潜在问题和改进建议。"
            "传入要审查的文件路径或模块名称。"
            "返回结构化的审查报告。"
        ),
        system_prompt=(
            "你是一个严格的代码审查员。分析代码质量、潜在 bug、安全问题、性能问题。\n"
            "返回结构化的审查报告，格式如下：\n"
            "\n"
            "## [严重问题]\n"
            "（会导致 bug 或安全漏洞的问题）\n"
            "\n"
            "## [改进建议]\n"
            "（可以改进但不影响功能的地方）\n"
            "\n"
            "## [代码亮点]\n"
            "（写得好的地方，值得保留或推广）\n"
            "\n"
            "每个条目都要指出具体的文件和行号。如果代码没有问题，直接说\'未发现严重问题\'。"
        ),
        allowed_tools=["read_file", "glob"],
        max_turns=5,
        timeout=90.0,
    )
    return SubAgent(config, raw_client, model, parent_registry, cwd)
'''

with open("../src/sub_agents.py", "w", encoding="utf-8") as f:
    f.write(sub_agents_code)

print("已保存: src/sub_agents.py")

# 验证文件可以导入（语法检查）
import subprocess
result = subprocess.run(
    ["python", "-c", "import ast; ast.parse(open('../src/sub_agents.py').read()); print('语法检查通过')"],
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

## 小结

| 模块 | 作用 |
|------|------|
| `SubAgentConfig` | 子代理配置：工具白名单、最大回合数、超时 |
| `LoopEvent` / `EventKind` | 循环内部的事件流，供调用方监听 |
| `agentic_loop_v2` | 通用 Agentic Loop，主代理和子代理共用 |
| `SubAgent` | 把完整的 Agentic Loop 封装成一个 `Tool`，暴露单一参数 `task` |
| `create_codebase_investigator` | 只读工具（read/list/glob），专注调查代码库结构 |
| `create_code_reviewer` | 只读工具（read/glob），返回结构化 code review 报告 |

**关键设计决策**：

- 子代理复用父代理的 `raw_client`（同一个 LLM 连接），不新建 HTTP 会话
- 子代理用独立的 `ContextManager`，调查完销毁，主代理 context 不增长
- `allowed_tools` 在执行时才从父 registry 筛选，而不是构造时——这样父 registry 可以在子代理创建后继续注册工具
- 超时用 `asyncio.timeout`，子代理超时不会卡死主代理

下一章实现**并行子代理**：主代理同时派出多个子代理，用 `asyncio.gather` 并行执行，然后合并结果。

In [ ]:
await raw_client.close()
print("连接已关闭")